# Atividade Prática — Análise Preditiva e Prescritiva

## MegaVendas Brasil: previsão de lucro e recomendação de preço

A MegaVendas Brasil é uma rede de varejo multicanal com mais de 300 pontos de venda físicos,
e-commerce e aplicativo próprio. A empresa quer usar o histórico de vendas para duas coisas:

| Etapa | Pergunta |
|---|---|
| **Análise preditiva** | Quanto lucro esta venda provavelmente vai gerar? |
| **Análise prescritiva** | Que combinação de preço e volume maximiza o lucro? |

**O que você precisa entregar**

1. Comparar **seis** modelos de regressão prevendo o lucro, todos sobre os mesmos dados.
2. Uma tabela única com R², RMSE e MAE dos seis, ordenada do melhor para o pior.
3. O gráfico de valores reais contra previstos do modelo vencedor.
4. As variáveis de maior influência, com sua interpretação.
5. Simulação de cenários variando quantidade e preço, em mapa de calor.
6. Recomendações de preço, volume e campanha, e o efeito de uma restrição de negócio.

> **A interpretação em markdown vale tanto quanto o código.** Um notebook que roda e não
> explica o que os números significam não cumpre o objetivo da atividade.
>
> As células marcadas com `# TODO` são suas. Os textos em markdown dizem o que se espera
> de cada etapa — substitua-os pela sua própria leitura dos resultados.

## 1. Instalação das dependências

`pandas`, `numpy`, `matplotlib` e `scikit-learn` costumam já vir instalados.
XGBoost, LightGBM e CatBoost **não vêm no Anaconda padrão**.

A célula abaixo instala o que estiver faltando. Se a instalação não for possível no seu
ambiente, o notebook continua rodando e o modelo ausente fica fora da comparação.

In [ ]:
import importlib, subprocess, sys

for pacote in ["xgboost", "lightgbm", "catboost"]:
    try:
        importlib.import_module(pacote)
        print(f"{pacote}: já instalado")
    except ImportError:
        print(f"{pacote}: instalando...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pacote], check=False)

## 2. Importação das bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

DISPONIVEL = {}
try:
    from xgboost import XGBRegressor
    DISPONIVEL["XGBoost"] = True
except ImportError:
    DISPONIVEL["XGBoost"] = False
try:
    from lightgbm import LGBMRegressor
    DISPONIVEL["LightGBM"] = True
except ImportError:
    DISPONIVEL["LightGBM"] = False
try:
    from catboost import CatBoostRegressor
    DISPONIVEL["CatBoost"] = True
except ImportError:
    DISPONIVEL["CatBoost"] = False

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.3f}".format)

for nome, ok in DISPONIVEL.items():
    print(f"{nome}: {'disponível' if ok else 'AUSENTE — ficará fora da comparação'}")

## 3. Carregamento da base

> **Atenção.** O arquivo usa **ponto e vírgula** como separador e **vírgula** como decimal.
> Sem `sep=";"` e `decimal=","` o pandas lê tudo como uma única coluna de texto.

In [ ]:
df = pd.read_csv("vendas_rede_varejo.csv", sep=";", decimal=",")

print(f"Linhas: {df.shape[0]}   Colunas: {df.shape[1]}")
df.head()

In [ ]:
df.info()
print("\nValores ausentes:", df.isnull().sum().sum())

## 4. Criação da variável-alvo

O lucro de cada venda é o que sobra depois do custo:

> **lucro = valor_total − custo_total**

In [ ]:
df["lucro"] = df["valor_total"] - df["custo_total"]

print(df["lucro"].describe().to_string())
print(f"\nMargem média: {(df['lucro'] / df['valor_total']).mean():.1%}")

### Distribuição do lucro

Antes de modelar, veja a forma do alvo. Monte um histograma e um boxplot do lucro e
escreva o que eles mostram: a distribuição é simétrica? Há valores extremos?

In [ ]:
# TODO: histograma e boxplot do lucro
# Sugestão: plt.subplots(1, 2, figsize=(12, 4)), hist() no primeiro eixo e boxplot() no segundo.

**Sua interpretação:** descreva a forma da distribuição. A média e a mediana coincidem?
O que isso diz sobre a cauda?

# PARTE 1 — ANÁLISE PREDITIVA

## Pergunta central

> **Quanto lucro esta venda provavelmente vai gerar?**

O alvo é um valor contínuo em reais, então é um problema de **regressão**.

## 5. Seleção das variáveis

As variáveis explicativas são **`quantidade`**, **`preco_unitario`** e **`campanha`**.

> ### Por que `valor_total` e `custo_total` ficam de fora
>
> O lucro é calculado a partir dessas duas colunas. Usá-las como entrada entrega a resposta
> ao modelo: ele não aprende nada e o R² vai a 1,000. Isso se chama **vazamento de dados**,
> e invalida qualquer comparação.
>
> Na prática há outra razão: no momento de decidir o preço, o custo total daquela venda
> ainda não aconteceu. Um modelo que depende dele não pode ser usado para decidir.

In [ ]:
features = ["quantidade", "preco_unitario", "campanha"]
alvo = "lucro"

X = df[features]
y = df[alvo]

X.head()

### Demonstração do vazamento (opcional, mas instrutivo)

Treine rapidamente um modelo **incluindo** `valor_total` e `custo_total` e observe o R².
Depois apague essas colunas e siga o resto da atividade sem elas.

In [ ]:
# TODO (opcional): treine um modelo incluindo valor_total e custo_total e veja o R².

## 6. Separação entre treino e teste

A separação vem **antes** de qualquer treino. O conjunto de teste é tocado uma única vez,
no fim, para medir o desempenho.

- 70% para treino, 30% para teste;
- `random_state=42` para o resultado ser reproduzível.

In [ ]:
# TODO: separe treino e teste
# - test_size=0.30
# - random_state=42
# X_train, X_test, y_train, y_test = ...

## 7. Preparação dos dados

`campanha` é texto e precisa virar número. Usamos **One-Hot Encoding**: cada campanha
vira uma coluna de 0 e 1. As duas variáveis numéricas passam direto.

O preparador entra dentro de um `Pipeline`, junto com o modelo. Assim a mesma transformação
é aplicada ao treino e ao teste, sem risco de vazamento.

In [ ]:
# TODO: monte o ColumnTransformer
# - OneHotEncoder(handle_unknown="ignore") para ["campanha"]
# - remainder="passthrough" para quantidade e preco_unitario
# preprocessador = ...

## 8. Os seis modelos

| Modelo | Família | Em uma frase |
|---|---|---|
| Regressão Linear | Linear | Ajusta uma soma ponderada das variáveis |
| Árvore de Decisão | Árvore | Divide os dados em faixas e prevê a média de cada folha |
| Random Forest | Bagging | Muitas árvores diferentes, previsão é a média delas |
| XGBoost | Boosting | Árvores em sequência, cada uma corrigindo o erro da anterior |
| LightGBM | Boosting | Boosting otimizado para velocidade |
| CatBoost | Boosting | Boosting desenhado para variáveis categóricas |

In [ ]:
# TODO: monte um dicionário com os seis modelos, cada um dentro de um Pipeline
# com o preprocessador. Proteja os três boostings com `if DISPONIVEL[...]`.
#
# modelos = {}
# modelos["Regressão Linear"] = Pipeline([("pre", preprocessador), ("modelo", LinearRegression())])
# ...

## 9. Treino e avaliação dos seis

Todos treinam no mesmo `X_train` e são medidos no mesmo `X_test`, com as mesmas três métricas:

- **R²** — quanto da variação do lucro o modelo explica (quanto mais perto de 1, melhor);
- **RMSE** — erro médio em reais, penalizando os erros grandes;
- **MAE** — erro médio em reais, tratando todos os erros igual.

In [ ]:
# TODO: treine cada modelo no conjunto de treino e avalie no de teste.
# Monte um DataFrame `comparacao` com as colunas Modelo, R², RMSE e MAE,
# ordenado do melhor R² para o pior.
#
# Dica: np.sqrt(mean_squared_error(y_test, pred)) devolve o RMSE.

## 10. Comparação visual

In [ ]:
# TODO: gráfico de barras comparando R² e RMSE dos seis modelos.

**Sua interpretação:** compare os seis.

- Qual família de modelo se saiu melhor, e por quê?
- A Regressão Linear ficou atrás? Pense na forma da relação entre lucro, quantidade e preço.
- O RMSE é maior que o MAE? O que isso diz sobre onde o modelo erra mais?

## 11. O modelo vencedor

In [ ]:
# TODO: selecione o modelo de melhor R² e guarde em `melhor`.
# melhor_nome = ...
# melhor = modelos[melhor_nome]
# y_pred = previsoes[melhor_nome]

## 12. Valores reais contra valores previstos

Cada ponto é uma venda do conjunto de teste. A linha diagonal é o acerto perfeito:
quanto mais perto dela, melhor a previsão.

In [ ]:
# TODO: gráfico de dispersão de y_test contra y_pred, com a diagonal de acerto perfeito.

**Sua interpretação:** os pontos acompanham a diagonal? Em que faixa de lucro o modelo
erra mais, e por que isso acontece?

## 13. Variáveis de maior influência

In [ ]:
# TODO: extraia a importância das variáveis do modelo vencedor e plote.
# Atenção: modelos de árvore têm .feature_importances_; a Regressão Linear tem .coef_.

**Sua interpretação:** quais variáveis o modelo mais usou? A soma das campanhas é
relevante perto de quantidade e preço? O que isso sugere sobre o efeito das campanhas?

# PARTE 2 — ANÁLISE PRESCRITIVA

## Pergunta central

> **Que combinação de preço e volume maximiza o lucro?**

Saímos de "quanto vai dar" para "o que devo fazer". O modelo da Parte 1 vira a ferramenta
de simulação: montamos cenários que não estão na base e perguntamos a ele o resultado.

## 14. Grade de cenários

Variamos `quantidade` e `preco_unitario` dentro da faixa observada na base e pedimos ao
modelo o lucro previsto de cada combinação.

In [ ]:
# TODO: monte a grade de cenários
# - quantidade de 2 a 18, de 2 em 2
# - preco_unitario em 8 valores entre 300 e 2400
# - campanha fixa em "Nenhuma" (vamos variá-la depois)
# Preveja o lucro de cada combinação e organize em uma matriz com pivot().

## 15. Mapa de calor do lucro previsto

Os eixos levam os **valores reais** de quantidade e preço — não os índices da matriz.

In [ ]:
# TODO: mapa de calor do lucro previsto, com imshow() ou heatmap().
# Exigências:
# - eixos rotulados com os valores reais de quantidade e preço, não com índices
# - barra de cor com legenda
# - destaque do melhor cenário

**Sua interpretação:** leia o mapa.

- Qual combinação de preço e volume maximiza o lucro previsto?
- O gradiente é suave ou tem patamares? Se tiver, o que isso diz sobre como o modelo prevê?
- Mais quantidade sempre aumenta a previsão? Se não, por que isso acontece?

## 16. A campanha faz diferença?

O enunciado pede para recomendar qual campanha sustenta o melhor resultado.
Antes de recomendar, é preciso verificar se existe diferença.

In [ ]:
# TODO: para cada campanha, repita a mesma grade trocando só a campanha
# e compare o lucro previsto médio. Depois compare com a margem observada
# por campanha na base: a diferença entre campanhas é maior que o desvio
# padrão dentro de cada uma?

**Sua interpretação:** a campanha faz diferença no lucro previsto?

- Qual é a diferença entre a melhor e a pior campanha, em reais e em percentual?
- Compare essa diferença com o desvio padrão da margem **dentro** de cada campanha.
- Com base nisso, dá para recomendar uma campanha? Justifique.

## 17. Restrição de negócio

Recomendação sem restrição não é decisão: basta escolher o máximo. O que torna o problema
real é o que não se pode fazer.

Cenário: **a política comercial não permite preço unitário acima de R$ 1.500.**

In [ ]:
# TODO: aplique uma restrição realista de negócio e refaça a recomendação.
# Exemplos: teto de preço por política comercial, limite de estoque por venda,
# ou volume máximo que a logística entrega.
# Compare o melhor cenário com e sem a restrição e quantifique o custo dela.

**Sua interpretação:** qual restrição você escolheu e por quê?

- Qual passa a ser a melhor combinação sob a restrição?
- Quanto ela custa, em reais e em percentual?
- Como você apresentaria esse número para a área de negócio?

## 18. Recomendações finais

Escreva aqui suas recomendações para a MegaVendas Brasil, cobrindo:

1. **Preço e volume** — qual faixa recomendar, e quais são os limites dessa recomendação.
2. **Campanha** — o que os dados permitem (ou não permitem) afirmar.
3. **Restrição** — como a restrição que você escolheu muda a decisão, e quanto custa.
4. **Limites do modelo** — o que ele não sabe, e o que seria preciso para saber.

Lembre-se da diferença entre associação e causa.

# Checklist de entrega

Confira antes de enviar:

- [ ] Base lida com `sep=";"` e `decimal=","`
- [ ] Coluna `lucro` criada, sem usar `valor_total` nem `custo_total` como entrada
- [ ] Treino e teste separados antes de qualquer treino, com `random_state` fixo
- [ ] Seis modelos treinados sobre os mesmos dados
- [ ] Tabela única com R², RMSE e MAE, ordenada
- [ ] Gráfico de valores reais contra previstos do modelo vencedor
- [ ] Variáveis de maior influência, com interpretação
- [ ] Grade de cenários e mapa de calor com eixos em valores reais
- [ ] Recomendação de preço, volume e campanha
- [ ] Restrição de negócio e seu custo
- [ ] Interpretação em markdown em cada etapa